# Step 4 — FAISS

Replace the hand-written search loop with a real vector index, and **persist it
to disk** so the app never re-embeds on startup.

FAISS (Facebook AI Similarity Search) is free, local, and needs no server.

## Setup — rebuild chunks, reuse cached vectors

No embedding API calls here: the vectors come from `vectors_cache.npz` made in notebook 02.

In [ ]:
import os, warnings, requests, numpy as np
warnings.filterwarnings("ignore")          # hide the langchain-community sunset notice

from dotenv import load_dotenv
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Put your key in rebuild/.env"

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; learning-project)"}

def load_url(url: str) -> Document:
    r = requests.get(url, headers=HEADERS, timeout=30); r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
    main = soup.find("main") or soup.find("article") or soup.body
    return Document(
        page_content=main.get_text(separator="\n", strip=True),
        metadata={"source": url,
                  "title": soup.title.get_text(strip=True) if soup.title else url},
    )

URLS = [
    "https://en.wikipedia.org/wiki/Tata_Motors",
    "https://en.wikipedia.org/wiki/Mahindra_%26_Mahindra",
]
docs = [load_url(u) for u in URLS]
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""], chunk_size=1000, chunk_overlap=200
)
chunks = splitter.split_documents(docs)

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")
chunk_vectors = np.load("vectors_cache.npz")["v"].tolist()

assert len(chunks) == len(chunk_vectors), "Re-run notebook 02 — cache is stale"
print(f"{len(chunks)} chunks, {len(chunk_vectors)} cached vectors")

## 4.1 — Build the index without re-embedding

The obvious call is `FAISS.from_documents(chunks, embeddings)` — but that
**re-embeds all 108 chunks**, costing ~90 seconds and a chunk of your rate limit
every single run.

Since we already have the vectors, `from_embeddings` takes them directly.

> `langchain-community` is being sunset, but FAISS still lives there and has no
> maintained standalone replacement yet. It works; just expect it to move.

In [ ]:
from langchain_community.vectorstores import FAISS
import time

t0 = time.time()
store = FAISS.from_embeddings(
    text_embeddings=list(zip([c.page_content for c in chunks], chunk_vectors)),
    embedding=embeddings,                    # kept for embedding future queries
    metadatas=[c.metadata for c in chunks],
)
print(f"index built in {time.time()-t0:.2f}s with ZERO API calls")
print("vectors in index:", store.index.ntotal)
print("dimensions      :", store.index.d)

## 4.2 — Save and reload

This is the part that makes the app practical: process URLs once, answer
questions forever.

In [ ]:
store.save_local("faiss_index")
print("files written:", sorted(os.listdir("faiss_index")))

Two files:

- **`index.faiss`** — the vectors, in FAISS's binary format
- **`index.pkl`** — the chunk text and metadata, **pickled**

That pickle is why reloading demands an explicit flag. Unpickling runs arbitrary
code, so LangChain refuses unless you confirm you trust the file.

**This is exactly why we did not load the `faiss_store_openai.pkl` that shipped
with the original project.** Ours is safe because we just created it.

In [ ]:
reloaded = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True,   # safe: we wrote this file ourselves
)
print("reloaded vectors:", reloaded.index.ntotal)

## 4.3 — Search

`similarity_search_with_score` embeds the question (1 API call) and returns the
nearest chunks.

In [ ]:
results = reloaded.similarity_search_with_score(
    "Which company makes the Mahindra Thar?", k=3
)

for doc, score in results:
    src = doc.metadata["source"].split("/")[-1]
    print(f"[{score:.3f}] {src}")
    print("   ", doc.page_content[:160].replace("\n", " "), "...\n")

## ⚠️ The scores mean the OPPOSITE of notebook 02

In notebook 02 we used **cosine similarity**, where **higher = better** (0.82
was a strong match).

FAISS by default returns **L2 distance**, where **lower = better**. Look at the
scores above — they *ascend*, because the best match is listed first.

Getting this backwards is a classic bug: you sort the wrong way and hand the LLM
your *least* relevant chunks. Always check which metric your store returns.

In [ ]:
scores = [s for _, s in results]
print("scores in order returned:", [round(s, 3) for s in scores])
print("ascending (lower = closer)?", scores == sorted(scores))

## 4.4 — Does FAISS agree with our manual search?

Same question, both methods. The ranking should broadly match — FAISS is an
optimisation, not a different algorithm.

In [ ]:
import math

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(y*y for y in b)))

q = "What vehicles does Tata Motors produce?"
qv = embeddings.embed_query(q)

# brute force, as in notebook 02
t0 = time.time()
manual = sorted(((cosine(qv, cv), i) for i, cv in enumerate(chunk_vectors)), reverse=True)[:3]
t_manual = time.time() - t0

# FAISS, using the same query vector so no second API call
t0 = time.time()
faiss_hits = reloaded.similarity_search_with_score_by_vector(qv, k=3)
t_faiss = time.time() - t0

print(f"brute force: {t_manual*1000:.1f} ms")
print(f"FAISS      : {t_faiss*1000:.1f} ms\n")

print("brute-force top 3:")
for s, i in manual:
    print(f"   {chunks[i].page_content[:60]!r}")
print("\nFAISS top 3:")
for d, s in faiss_hits:
    print(f"   {d.page_content[:60]!r}")

At 108 chunks the speed difference is small — FAISS's advantage shows at
scale. Brute force is O(n): 100,000 chunks means 100,000 comparisons per query.
FAISS uses an index structure that skips most of them.

What you get *today* is **persistence**, which matters just as much.

## 4.5 — The retriever

For step 5 we need a **retriever** — a standard interface the chain calls to
fetch context.

In [ ]:
retriever = reloaded.as_retriever(search_kwargs={"k": 4})

found = retriever.invoke("Who founded Mahindra?")
print(f"retrieved {len(found)} chunks\n")
for d in found:
    print(" -", d.metadata["source"].split("/")[-1], "|",
          d.page_content[:80].replace("\n", " "), "...")

---

## Checkpoint

- Index built from cached vectors, **zero API calls**
- Saved to `faiss_index/`, reloadable without re-embedding
- A `retriever` ready to plug into a chain

**Next:** wire Gemini in so it answers questions *using* these chunks, and cites
which article each answer came from.